# HuggingFace Datasets and DataLoader for LLM Training

For LLM fine-tuning, data preparation is 50% of the work.
This notebook covers:
- Loading and processing datasets with HuggingFace `datasets`
- Tokenizing text for causal LM training
- Building a DataLoader from tokenized data
- Data formats for SFT, DPO, and GRPO

**Install:** `pip install datasets transformers`

In [ ]:
from datasets import Dataset, load_dataset
from transformers import AutoTokenizer
import torch
from torch.utils.data import DataLoader

# --- Load a dataset from HuggingFace Hub ---
# 'trl-lib/ultrachat_200k' is a common SFT dataset (instruction-response pairs)
# Using a tiny sample here for speed

# ds = load_dataset('trl-lib/ultrachat_200k', split='train[:1%]')  # 1% of training set
# print(ds)
# print(ds[0])   # see what a sample looks like

# For now, create a synthetic dataset
raw_data = [
    {"instruction": "What is PyTorch?",       "response": "PyTorch is a deep learning framework built on tensors."},
    {"instruction": "What is fine-tuning?",   "response": "Fine-tuning adapts a pre-trained model to a specific task."},
    {"instruction": "What is LoRA?",          "response": "LoRA is a parameter-efficient fine-tuning method that trains low-rank adapters."},
    {"instruction": "What is backprop?",      "response": "Backpropagation computes gradients by applying the chain rule backward through the network."},
    {"instruction": "What is a transformer?", "response": "A transformer uses self-attention to process sequences in parallel."},
]

dataset = Dataset.from_list(raw_data)
print(dataset)
print(dataset[0])

## Formatting for different training objectives

### SFT format — single turn
Tokenize (instruction + response) as one sequence. Model learns to predict the response tokens.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained('gpt2')
tokenizer.pad_token = tokenizer.eos_token

def format_sft(example):
    """Format instruction + response into a single training string."""
    return {
        'text': f"<|user|>\n{example['instruction']}\n<|assistant|>\n{example['response']}{tokenizer.eos_token}"
    }

# Apply formatting
formatted = dataset.map(format_sft)
print('Formatted example:')
print(formatted[0]['text'])

In [ ]:
def tokenize(example):
    """Tokenize text and create labels = input_ids (for causal LM loss)."""
    encoded = tokenizer(
        example['text'],
        truncation=True,
        max_length=256,
        padding='max_length',   # pad all sequences to same length
    )
    encoded['labels'] = encoded['input_ids'].copy()   # label = input shifted by 1 (handled by model)
    return encoded

tokenized = formatted.map(tokenize, remove_columns=['instruction', 'response', 'text'])
tokenized.set_format('torch')   # return torch tensors instead of lists

print('Columns:', tokenized.column_names)
print('input_ids shape:', tokenized[0]['input_ids'].shape)
print('labels shape:   ', tokenized[0]['labels'].shape)

In [ ]:
# --- DataLoader from HuggingFace Dataset ---
from torch.utils.data import DataLoader

loader = DataLoader(tokenized, batch_size=2, shuffle=True)

batch = next(iter(loader))
print('Batch keys:         ', list(batch.keys()))
print('input_ids shape:    ', batch['input_ids'].shape)    # [2, 256]
print('attention_mask shape:', batch['attention_mask'].shape)
print('labels shape:        ', batch['labels'].shape)

## DPO data format — preference pairs

In [ ]:
# DPO requires (prompt, chosen, rejected) triples
dpo_data = [
    {
        'prompt':   'How do I sort a list in Python?',
        'chosen':   'Use list.sort() for in-place sorting, or sorted(list) to return a new sorted list.',
        'rejected': 'Just Google it.'
    },
    {
        'prompt':   'Explain gradient descent.',
        'chosen':   'Gradient descent updates weights by moving in the direction that minimizes the loss, guided by gradients.',
        'rejected': 'It is a way to train models.'
    },
]

dpo_dataset = Dataset.from_list(dpo_data)
print('DPO dataset:')
print(dpo_dataset[0])

# TRL's DPOTrainer accepts this format directly
# It tokenizes chosen and rejected internally

## GRPO data format — prompts only

GRPO just needs prompts. It generates responses at training time and scores them with your reward function.

In [ ]:
grpo_data = [
    {'prompt': 'What is 15 × 7?'},
    {'prompt': 'What is 144 / 12?'},
    {'prompt': 'What is 23 + 45?'},
]

grpo_dataset = Dataset.from_list(grpo_data)
print('GRPO dataset (prompts only):')
print(grpo_dataset[0])

# The reward function you write determines what "good" looks like:
# def reward_fn(completions, prompts, **kwargs):
#     rewards = []
#     for completion, prompt in zip(completions, prompts):
#         expected = eval(prompt.split('?')[0].split('What is ')[1])  # compute answer
#         is_correct = str(expected) in completion
#         rewards.append(1.0 if is_correct else 0.0)
#     return rewards

## Data preparation checklist for LLM fine-tuning

```
□ Correct format for your training method (SFT/DPO/GRPO)
□ Tokenized with the model's tokenizer (not a different one)
□ Max sequence length set (common: 512-2048 for fine-tuning)
□ Attention mask included (tells model which tokens are padding)
□ Labels set correctly (SFT: same as input_ids; padding positions set to -100)
□ Shuffle training data, don't shuffle validation
□ num_workers=0 on Windows (avoids multiprocessing issues)
□ pin_memory=True if using GPU (faster CPU→GPU transfer)
```